# 11.19 Soft Actor-Critic

SAC treats randomness as a feature of the objective, not merely training noise.

Soft Actor-Critic uses the RL return machinery while adding entropy so robust exploration is part of optimization. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1119
random.seed(SEED)
np.random.seed(SEED)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * float(reward)
        power = power * gamma
    return total


lesson_rewards = [1, 0, 2]
lesson_gamma = 0.9
lesson_return = discounted_return(lesson_rewards, lesson_gamma)
assert abs(lesson_return - 2.62) < 1e-12


ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["up", "right", "down", "left"]
DELTAS = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}


def make_grid_env(name, height, width, start, goal, walls=None, slip=0.0, wind=None, sparse=True, horizon=35, obs_noise=0.0):
    walls = set(walls or [])
    wind = wind or {}
    states = []
    state_index = {}
    for row in range(height):
        for col in range(width):
            cell = (row, col)
            if cell not in walls:
                state_index[cell] = len(states)
                states.append(cell)
    env = {
        "name": name,
        "height": height,
        "width": width,
        "start": start,
        "goal": goal,
        "walls": walls,
        "slip": float(slip),
        "wind": wind,
        "sparse": bool(sparse),
        "horizon": int(horizon),
        "states": states,
        "state_index": state_index,
        "n_states": len(states),
        "n_actions": 4,
        "obs_noise": float(obs_noise),
    }
    return env


def step_cell(env, cell, action, rng):
    actual = int(action)
    if rng.random() < env["slip"]:
        actual = int(rng.integers(0, 4))
    row_delta, col_delta = DELTAS[actual]
    row = cell[0] + row_delta
    col = cell[1] + col_delta
    pushed = env["wind"].get((cell[0], cell[1]), (0, 0))
    row = row + pushed[0]
    col = col + pushed[1]
    candidate = (max(0, min(env["height"] - 1, row)), max(0, min(env["width"] - 1, col)))
    if candidate in env["walls"]:
        candidate = cell
    reward = -0.02
    done = candidate == env["goal"]
    if done:
        reward = 1.0
    if not env["sparse"]:
        distance = abs(candidate[0] - env["goal"][0]) + abs(candidate[1] - env["goal"][1])
        reward = reward - 0.01 * distance
    return candidate, reward, done


def transition_table(env):
    n_states = env["n_states"]
    n_actions = env["n_actions"]
    transitions = np.zeros((n_states, n_actions, n_states), dtype=float)
    rewards = np.zeros((n_states, n_actions, n_states), dtype=float)
    rng = np.random.default_rng(SEED + env["n_states"])
    samples = 80
    for state_id, cell in enumerate(env["states"]):
        for action in range(n_actions):
            for _ in range(samples):
                next_cell, reward, done = step_cell(env, cell, action, rng)
                next_id = env["state_index"][next_cell]
                transitions[state_id, action, next_id] = transitions[state_id, action, next_id] + 1.0
                rewards[state_id, action, next_id] = rewards[state_id, action, next_id] + reward
            total = transitions[state_id, action].sum()
            mask = transitions[state_id, action] > 0
            rewards[state_id, action, mask] = rewards[state_id, action, mask] / transitions[state_id, action, mask]
            transitions[state_id, action] = transitions[state_id, action] / total
    return transitions, rewards


def build_f12_env_ladder():
    ladder = [
        make_grid_env("D1 two-state chain", 1, 2, (0, 0), (0, 1), horizon=6, sparse=False),
        make_grid_env("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), slip=0.15, horizon=10, sparse=False),
        make_grid_env("D3 4x4 gridworld", 4, 4, (0, 0), (3, 3), walls={(1, 1), (2, 1)}, horizon=25, sparse=False),
        make_grid_env("D4 stochastic windy grid", 5, 5, (0, 0), (4, 4), walls={(1, 2), (2, 2)}, slip=0.2, wind={(3, 1): (-1, 0), (3, 2): (-1, 0)}, horizon=35, sparse=False),
        make_grid_env("D5 larger sparse grid", 7, 7, (0, 0), (6, 6), walls={(1, 1), (1, 2), (2, 4), (3, 1), (3, 2), (4, 4), (5, 2)}, slip=0.25, wind={(5, 3): (-1, 0)}, horizon=45, sparse=True),
    ]
    assert len(ladder) == 5
    assert [env["n_states"] for env in ladder] == [2, 3, 14, 23, 42]
    return ladder


def greedy_rollout(env, policy, episodes=20):
    rng = np.random.default_rng(SEED + env["height"] + env["width"])
    returns = []
    paths = []
    for episode in range(episodes):
        cell = env["start"]
        rewards = []
        path = [cell]
        for step in range(env["horizon"]):
            state_id = env["state_index"][cell]
            action = int(policy[state_id])
            cell, reward, done = step_cell(env, cell, action, rng)
            rewards.append(reward)
            path.append(cell)
            if done:
                break
        returns.append(discounted_return(rewards, lesson_gamma))
        paths.append(path)
    return float(np.mean(returns)), paths[-1]


def plot_grid_values(ax, env, values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    for cell, idx in env["state_index"].items():
        grid[cell] = values[idx]
    image = ax.imshow(grid, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return image


## The concept, built once: reward plus entropy\n\nThe soft objective backs up return and policy entropy together. For a tabular state, use\n$$V_{soft}(s)=\alpha\log\sum_a\exp(Q(s,a)/\alpha),\qquad \pi(a\mid s)=\frac{\exp(Q(s,a)/\alpha)}{\sum_b\exp(Q(s,b)/\alpha)}.$$\nThe lesson's discounted numbers are still present: $G=1+0.9\cdot0+0.9^2\cdot2=2.620$.

In [ ]:

def soft_policy_update(q_values, alpha=0.2):
    scaled = np.asarray(q_values, dtype=float) / alpha
    shifted = scaled - np.max(scaled)
    weights = np.exp(shifted)
    policy = weights / weights.sum()
    soft_value = alpha * (np.log(np.exp(shifted).sum()) + np.max(scaled))
    entropy = -float(np.sum(policy * np.log(policy + 1e-12)))
    return policy, float(soft_value), entropy


q_values = np.array([1.0, 1.8])
policy, soft_value, entropy = soft_policy_update(q_values, alpha=0.2)
print("policy", np.round(policy, 6), "soft value", round(soft_value, 6), "entropy", round(entropy, 6))
assert abs(lesson_return - 2.62) < 1e-12
assert np.allclose(policy, np.array([0.01798621, 0.98201379]), atol=1e-6)
assert abs(soft_value - 1.803629) < 1e-6


Now make it a reusable tabular SAC-style planner. The Bellman target keeps future return and the entropy-smoothed next value instead of optimizing immediate reward only.

In [ ]:

def soft_value_iteration(env, alpha=0.2, gamma=0.9, iterations=80):
    transitions, rewards = transition_table(env)
    q_values = np.zeros((env["n_states"], env["n_actions"]))
    for _ in range(iterations):
        soft_values = np.array([soft_policy_update(q_values[state], alpha)[1] for state in range(env["n_states"])])
        next_q = np.zeros_like(q_values)
        for state in range(env["n_states"]):
            for action in range(env["n_actions"]):
                target = rewards[state, action] + gamma * soft_values
                next_q[state, action] = np.sum(transitions[state, action] * target)
        q_values = next_q
    policies = np.zeros_like(q_values)
    entropies = np.zeros(env["n_states"])
    for state in range(env["n_states"]):
        policies[state], _, entropies[state] = soft_policy_update(q_values[state], alpha)
    greedy_policy = np.argmax(policies, axis=1)
    values = np.max(q_values, axis=1)
    return values, policies, entropies, greedy_policy


## The dataset ladder

Family F12 uses the inline D1-D5 sequential-decision ladder: 2-state chain, slippery 3-state, 4x4 grid, windy grid, and larger sparse grid.

In [ ]:

ladder = build_f12_env_ladder()
for env in ladder:
    sample_states = env["states"][:5]
    print(env["name"], "states", env["n_states"], "actions", env["n_actions"], "sample", sample_states)


In [ ]:

results = []
artifacts = []
for env in ladder:
    values, policies, entropies, greedy_policy = soft_value_iteration(env, alpha=0.2)
    mean_return, path = greedy_rollout(env, greedy_policy)
    results.append({"rung": env["name"], "return": mean_return, "entropy": float(np.mean(entropies))})
    artifacts.append((env, entropies, path))
print("rung | return | mean_entropy")
for row in results:
    print(row["rung"], round(row["return"], 3), round(row["entropy"], 3))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, entropies, path) in enumerate(artifacts):
    plot_grid_values(axes[0, col], env, entropies, env["name"])
axes[1, 0].plot([idx + 1 for idx in range(len(results))], [row["return"] for row in results], marker="o")
axes[1, 0].set_title("return vs rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: confusing reward with return

Dropping entropy and discounted future value chases immediate reward. The fix restores the soft return objective.

In [ ]:

d5 = ladder[-1]
transitions, rewards = transition_table(d5)
immediate_policy = np.argmax(np.sum(transitions * rewards, axis=2), axis=1)
wrong_return, _ = greedy_rollout(d5, immediate_policy)
values, policies, entropies, soft_policy = soft_value_iteration(d5, alpha=0.2)
fixed_return, _ = greedy_rollout(d5, soft_policy)
print("wrong immediate reward policy", round(wrong_return, 3))
print("fixed soft return policy", round(fixed_return, 3))


## Evaluate it + Practice

- Metric: compare D1-D5 return against a no-skill baseline.
- Sanity check: D1 should solve by hand and match the exact assertion in the build-once cell.
- Ablation: turn off the key idea and verify the metric worsens on D4 or D5.
- Failure signal: curves that look good only because support, entropy, shape, or rollout bias was hidden.

Practice 1: change one ladder parameter and predict the metric direction before running.


Practice 2: lower alpha toward zero and inspect how the entropy heatmaps change.